# Notebook 04 — Train YOLOv8 Crop Classifier → Export to ONNX

Trains a YOLOv8 classification model on Zimbabwe crop images and exports it to ONNX for production inference via ONNX Runtime (no PyTorch in prod).

In [ ]:
import os, json, shutil, hashlib, time
from pathlib import Path

DATA_DIR = Path(os.getenv('CROP_DATA_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\data\crops'))
EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
RUNS_DIR = Path(os.getenv('RUNS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\runs'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')
BASE_MODEL = os.getenv('CROP_BASE_MODEL', 'yolov8n-cls.pt')
EPOCHS = int(os.getenv('CROP_EPOCHS', '50'))
IMG_SIZE = int(os.getenv('CROP_IMG_SIZE', '224'))
BATCH = int(os.getenv('CROP_BATCH', '32'))
MIN_IMAGES_PER_CLASS = int(os.getenv('MIN_IMAGES_PER_CLASS', '10'))
MIN_TOP1_ACCURACY = float(os.getenv('MIN_CROP_TOP1_ACCURACY', '0.70'))

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

CROP_CLASSES = [
    'maize', 'mango', 'tomato', 'soya_beans', 'groundnuts',
    'tobacco', 'cotton', 'cabbage', 'potato', 'onion',
    'sugar_beans', 'sunflower',
]

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print('Crop classifier configuration')
print('DATA_DIR:', DATA_DIR)
print('EXPORTS_DIR:', EXPORTS_DIR)
print('MODEL_VERSION:', MODEL_VERSION)
print('Classes:', CROP_CLASSES)

Classes (12): ['maize', 'mango', 'tomato', 'soya_beans', 'groundnuts', 'tobacco', 'cotton', 'cabbage', 'potato', 'onion', 'sugar_beans', 'sunflower']


In [ ]:
train_dir = DATA_DIR / 'train'
val_dir = DATA_DIR / 'val'

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Crop dataset not found: {DATA_DIR}. Place your dataset under train/ and val/ class folders.')
if not train_dir.exists():
    raise FileNotFoundError(f'Missing crop training folder: {train_dir}')
if not val_dir.exists():
    raise FileNotFoundError(f'Missing crop validation folder: {val_dir}')

class_counts = {}
for cls in CROP_CLASSES:
    cls_dir = train_dir / cls
    images = []
    if cls_dir.exists():
        for pattern in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
            images.extend(cls_dir.glob(pattern))
    class_counts[cls] = len(images)

missing_or_small = {cls: n for cls, n in class_counts.items() if n < MIN_IMAGES_PER_CLASS}
print('Training image counts:', class_counts)
if missing_or_small:
    raise ValueError(f'Insufficient crop images per class. Need >= {MIN_IMAGES_PER_CLASS}: {missing_or_small}')

print('Crop dataset validation passed.')

Dataset path: \workspace\data\crops
Exists: False


In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=str(DATA_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=os.getenv('TRAIN_DEVICE', 'cpu'),
    project=str(RUNS_DIR),
    name='crop_classifier',
    exist_ok=True,
)
print('Training complete. Best weights:', results.save_dir)

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
best_pt = RUNS_DIR / 'crop_classifier' / 'weights' / 'best.pt'
if not best_pt.exists():
    raise FileNotFoundError(f'Best weights not found: {best_pt}')

best_model = YOLO(str(best_pt))
metrics = best_model.val(data=str(DATA_DIR), imgsz=IMG_SIZE, device=os.getenv('TRAIN_DEVICE', 'cpu'))
top1_acc = float(metrics.top1)
print(f'Validation Top-1 Accuracy: {top1_acc:.2%}')
assert top1_acc >= MIN_TOP1_ACCURACY, f'Accuracy {top1_acc:.2%} below {MIN_TOP1_ACCURACY:.0%} threshold — retrain with more data'

In [ ]:
export_path = Path(best_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True))
print('Exported ONNX to:', export_path)

dest = EXPORTS_DIR / 'crop_classifier_v1.onnx'
shutil.copy(export_path, dest)
print('Copied to export directory:', dest)

In [ ]:
meta = {
    'model': 'crop_classifier_v1',
    'version': MODEL_VERSION,
    'format': 'onnx',
    'source_notebook': '04_train_crop_classifier.ipynb',
    'dataset_dir': str(DATA_DIR),
    'dataset_class_counts': class_counts,
    'input_size': IMG_SIZE,
    'classes': CROP_CLASSES,
    'top1_accuracy': round(top1_acc, 4),
    'minimum_top1_accuracy': MIN_TOP1_ACCURACY,
    'sha256': sha256_file(dest),
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
meta_path = EXPORTS_DIR / 'crop_classifier_metadata.json'
meta_path.write_text(json.dumps(meta, indent=2))
print('Metadata saved:', meta_path)
print(json.dumps(meta, indent=2))

In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(dest), providers=['CPUExecutionProvider'])
dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: dummy})
print('ONNX smoke-test passed. Output shape:', out[0].shape)
print('crop_classifier_v1.onnx ready for production.')